# Tasks 0-9: poison / adapt / pocket-attack / oracle vs. meta_al detection / unlearn+TTA -- full pipeline, 4 architectures

Consolidated continual-learning pipeline covering **Task 0 through Task 9** of the real 10-task run,
repeated as **4 full independent runs** -- one each for a 1, 2, 3, and 4-hidden-layer classifier
architecture.

1. **Task 0** trains a classifier from scratch on task 0's own data -- no replay buffer (there's no
   prior task to replay from) and no poisoning (the real pipeline never poisoned task 0 either --
   confirmed empirically: `classifier_checkpoint_task0.pt`'s replay buffer has 0% `*_perturbed`
   entries). This is also where each run's architecture (`hidden_sizes`) gets fixed for good -- every
   later task just deep-copies the previous task's own model forward, so the architecture never needs
   to be re-specified after Task 0.
2. **Tasks 1-9** each chain every lineage forward from the *same* lineage's own state at the end of the
   previous task (not re-derived from a checkpoint) -- `clean_model`, `poisoned_model`, and the 6 fixed
   lineages (`dropped_oracle`, `amnesiac_oracle`, `opposite_oracle`, and the `meta_al` equivalents) all
   persist as live Python objects across the whole run. The replay buffer is still read fresh each task
   from the real checkpoint chain (`ckpts[task_id - 1]`, i.e. the end-of-previous-task state).
3. Each task crafts a **fresh** test-time pocket attack against **every** test set seen so far
   (task `N`'s cell attacks tasks `0..N`, not just task `N`).
4. Runs the two detectors -- **oracle** (supervised mixed-uncertainty XGBoost) and **meta_al**
   (active-learning episodic XGBoost, ~250-label budget) -- each through all three unlearning methods
   (`dropped-rows`, `amnesiac`, `opposite-class`).
5. Builds a 2%-of-genuine-pockets calibration probe from every task seen so far, combined, and applies
   test-time adaptation to the three `meta_al` models only (oracle models are **not** TTA'd).
6. Reports clean / adv / combined accuracy **and precision / recall / f1** (on the combined clean+adv
   set, malware=1 as positive class) for every task seen so far, plus the mean combined accuracy, for
   7 models: `poisoned (baseline)`, 3 oracle (no TTA), 3 meta_al+TTA -- plus Task 0's own single
   from-scratch model.
7. Saves every model's state_dict to `MODEL_SAVE_DIR_BASE/seed{SEED}_depth{depth}/task{N}/`, and writes
   one p/r/f1 CSV per architecture (plus one combined CSV across all 4) to `RESULTS_DIR`.

### Seeding
One `task_seed = SEED` is used for every task and every architecture run -- not derived per-task or
per-step. `torch.manual_seed(task_seed)` is called once at the top of each task's driver function, and
one shared `np.random.Generator(task_seed)` is created there and passed by reference into every
function that needs numpy randomness (including drawing each `XGBClassifier`'s `random_state` via
`rng.integers(...)`). Change `SEED` in the config cell and rerun for a fully independent realization of
the entire run (all 4 architectures, all 10 tasks).

### Caveat -- please read before running
This notebook has not been executed end-to-end in this session (no access to your checkpoint files or
`subsampled_dataset.h5` here) -- it was assembled from your previously-validated Task 1 logic,
generalized to Task 0-9 and to varying architectures. Please run it and paste back the first error if
anything breaks. Running all 4 architectures back to back is a large compute increase (4x the previous
single-architecture run) -- consider testing with `HIDDEN_SIZES_BY_DEPTH` trimmed to one entry first.


## 1. Config

In [23]:
import numpy as np
import torch
import torch.nn as nn
import copy
import json
import h5py
from pathlib import Path
from sklearn.metrics import classification_report, precision_recall_fscore_support, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import pandas as pd

# ---- EDIT THESE PATHS ----
H5_PATH = "subsampled_dataset.h5"
LOGS_DIR = Path("/home/aritran/ICLR/logs")
# ---------------------------

TASK_FRACTIONS = [
    0.3000, 0.0918, 0.0883, 0.0848, 0.0813,
    0.0778, 0.0743, 0.0708, 0.0673, 0.0638,
]
N_TASKS = len(TASK_FRACTIONS)

# ---- CHANGE THIS TO RUN THE WHOLE PIPELINE AGAIN WITH DIFFERENT RANDOMNESS ----
# task_seed = SEED -- the SAME seed is used for every task and every architecture (no per-task or
# per-step derivation). torch's global RNG is seeded once per task with task_seed and then left to
# evolve naturally through every step that follows (adaptation, unlearning, TTA -- no reseeding in
# between), and one shared np.random.Generator(task_seed) is passed into the detectors for their numpy
# randomness (including drawing the XGBoost random_state via rng.integers(...)). Bump SEED and rerun
# for a fully independent realization of the whole run.
SEED = 100
# ---------------------------------------------------------------------------------

POISON_FRACTION = 0.3
ADAPT_EPOCHS = 15
CLEAN_ADAPT_EPOCHS = 5
TASK0_EPOCHS = 30   # Task 0 trains from scratch (no prior model to continue from), so it gets more
                    # epochs than the incremental CLEAN_ADAPT_EPOCHS/ADAPT_EPOCHS used from Task 1 on

N_OUTER_EPISODES = 8
N_INNER_STEPS = 3
SAMPLES_PER_INNER_STEP = 4

N_PER_GROUP = 12
UNCERTAIN_FRACTION = 0.3

# NOTE: no random_state baked in here -- every XGBClassifier(...) call below passes its own seed
# drawn from the task's shared rng, instead of every task silently sharing a fixed random_state.
XGBOOST_PARAMS_BASE = dict(n_estimators=20, max_depth=2, learning_rate=0.1, reg_lambda=10.0,
                            subsample=0.7, colsample_bytree=0.3, min_child_weight=5,
                            eval_metric="logloss")

def xgb_params(seed):
    return dict(**XGBOOST_PARAMS_BASE, random_state=seed)

CALIB_FRAC = 0.02   # fraction of each task's genuine pockets held out for the calibration/TTA probe

# ---- architecture variants: 1/2/3/4 hidden layers, each a full independent Task 0-9 run ----
HIDDEN_SIZES_BY_DEPTH = {
    1: (512,),
    2: (512, 256),
    3: (512, 256, 128),
    4: (1024, 512, 256, 128),   # matches the real checkpoints' architecture
}

MODEL_SAVE_DIR_BASE = Path("./saved_models_datasensitivity")     # models saved under MODEL_SAVE_DIR_BASE/seed{SEED}_depth{depth}/
RESULTS_DIR = Path("./results_datasensitivity")                  # <-- EDIT to your actual target folder
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", DEVICE)


using device: cuda


## 2. Load the raw dataset (shared across all tasks)

In [2]:
with h5py.File(H5_PATH, "r") as f:
    features = f["features"][:]
    label = f["label"][:]

N = features.shape[0]
print("features:", features.shape, "label:", label.shape)


features: (234000, 1488) label: (234000,)


## 3. Checkpoint loading + gid reconstruction (generalized to any task_id)

In [3]:
def load_ckpt(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def get_ckpts_through(task_id, ckpts_all):
    """Loads (and caches) whatever checkpoints 0..task_id are needed. `ckpts_all` is a dict you
    keep around across task cells so earlier checkpoints aren't re-read from disk each time."""
    for t in range(task_id + 1):
        if t not in ckpts_all:
            p = LOGS_DIR / f"classifier_checkpoint_task{t}.pt"
            if not p.exists():
                raise FileNotFoundError(f"need {p} to reconstruct through task {task_id}")
            ckpts_all[t] = load_ckpt(p)
    return ckpts_all


def reconstruct_task_gids(max_task_id, ckpts):
    """Generalized version of the single-task boundary-correction logic (validated earlier on
    Task 1 alone): returns (task_train_gids, task_test_gids_final, INDEX_OFFSET) with keys 0..max_task_id.
    Only needs classifier_checkpoint_task{0..max_task_id}.pt, exactly like the original."""
    test_gids = {}
    for t, ck in ckpts.items():
        for task_id, gids in ck["task_test_gids"].items():
            gids = np.asarray(gids)
            if task_id not in test_gids or len(gids) > len(test_gids[task_id]):
                test_gids[task_id] = gids

    cum = np.cumsum(TASK_FRACTIONS)
    frac_upper = np.round(cum * N).astype(int)
    frac_upper[-1] = N
    frac_lower = np.concatenate([[0], frac_upper[:-1]])

    def out_of_bounds_count(offset):
        total = 0
        for g in test_gids.values():
            gi = g - offset
            total += ((gi < 0) | (gi >= N)).sum()
        return total

    def seam_mismatch_count(offset):
        total = 0
        for t, g in test_gids.items():
            gi = g - offset
            bad = (gi < frac_lower[t]) | (gi >= frac_upper[t])
            total += bad.sum()
        return total

    candidates = [0, 1]
    oob = {o: out_of_bounds_count(o) for o in candidates}
    seam = {o: seam_mismatch_count(o) for o in candidates}
    feasible = [o for o in candidates if oob[o] == 0]
    INDEX_OFFSET = min(feasible, key=lambda o: seam[o]) if feasible else min(candidates, key=lambda o: oob[o])
    print(f"  INDEX_OFFSET = {INDEX_OFFSET}  (out-of-bounds counts: {oob})")

    test_idx = {t: np.asarray(g) - INDEX_OFFSET for t, g in test_gids.items()}

    lower = frac_lower.copy()
    upper = frac_upper.copy()
    for t in range(1, N_TASKS):
        lo_side, hi_side = t - 1, t
        if lo_side not in test_idx or hi_side not in test_idx:
            continue
        constraint_lo = int(test_idx[lo_side].max()) + 1
        constraint_hi = int(test_idx[hi_side].min())
        if constraint_lo > constraint_hi:
            print(f"  !! seam {lo_side}/{hi_side}: irreconcilable -- investigate")
            continue
        boundary = min(max(frac_upper[lo_side], constraint_lo), constraint_hi)
        upper[lo_side] = boundary
        lower[hi_side] = boundary
    upper[-1] = N
    lower[0] = 0

    task_pool_idx = {t: set(range(int(lower[t]), int(upper[t]))) for t in range(N_TASKS)}

    task_train_gids, task_test_gids_final = {}, {}
    for t in range(max_task_id + 1):
        test_set = set(test_idx[t].tolist())
        missing = test_set - task_pool_idx[t]
        if missing:
            print(f"  task {t}: dropping {len(missing)} test indices outside computed pool")
            test_set -= missing
        train_set = task_pool_idx[t] - test_set
        task_train_gids[t] = np.array(sorted(train_set))
        task_test_gids_final[t] = np.array(sorted(test_set))
        print(f"  task {t}: train={len(train_set)} test={len(test_set)}")

    return task_train_gids, task_test_gids_final, INDEX_OFFSET


def rows_for(gids):
    idx = np.asarray(sorted(gids))
    return features[idx], label[idx], idx


def reconstruct_replay_buffer(buffer_list, offset):
    labels, categories, gids = zip(*buffer_list)
    gids = np.array(gids)
    idx = gids - offset
    X = features[idx].astype("float32")
    y = np.array(labels)
    categories = np.array(categories)
    n_poison = sum("perturbed" in c for c in categories)
    if n_poison:
        print(f"  note: {n_poison}/{len(buffer_list)} replay-buffer entries are *_perturbed -- "
              f"their X is the PRE-poison original feature vector, not the true perturbed one")
    return X, y, categories, gids


## 4. Model + adaptation wrapper (unchanged)

In [4]:
class ClassifierNN(nn.Module):
    """Matches the REAL checkpoints' key naming (layers.0-3 / bns.0-3 / fc_last) for the 4-layer
    architecture; for the 1/2/3-layer variants the same naming scheme just has fewer entries."""
    def __init__(self, input_dim, num_classes, hidden_sizes=(1024, 512, 256, 128)):
        super().__init__()
        sizes = [input_dim] + list(hidden_sizes)
        self.layers = nn.ModuleList([nn.Linear(sizes[i], sizes[i + 1]) for i in range(len(hidden_sizes))])
        self.bns = nn.ModuleList([nn.BatchNorm1d(h) for h in hidden_sizes])
        self.relu = nn.ReLU()
        self.fc_last = nn.Linear(hidden_sizes[-1], num_classes)

    def forward(self, x):
        for layer, bn in zip(self.layers, self.bns):
            x = self.relu(bn(layer(x)))
        return self.fc_last(x)


class AdaptableClassifier:
    def __init__(self, torch_model, lr=1e-4, weight_decay=1e-5):
        self.model = torch_model
        self.opt = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.loss_fn = nn.CrossEntropyLoss()

    def adapt(self, X, y, replay_X=None, replay_y=None, epochs=5, batch_size=128):
        Xt = torch.as_tensor(X, dtype=torch.float32)
        yt = torch.as_tensor(y, dtype=torch.long)
        if replay_X is not None and len(replay_X):
            Xt = torch.cat([Xt, torch.as_tensor(np.asarray(replay_X), dtype=torch.float32)], dim=0)
            yt = torch.cat([yt, torch.as_tensor(np.asarray(replay_y), dtype=torch.long)], dim=0)
        n = len(Xt)
        self.model.train()
        for _ in range(epochs):
            perm = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = perm[i:i + batch_size]
                if len(idx) < 2:
                    continue
                self.opt.zero_grad()
                loss = self.loss_fn(self.model(Xt[idx]), yt[idx])
                loss.backward()
                self.opt.step()
        self.model.eval()
        return self

    @torch.no_grad()
    def predict_proba(self, X):
        self.model.eval()
        logits = self.model(torch.as_tensor(X, dtype=torch.float32))
        return torch.softmax(logits, dim=1).numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def score(self, X, y):
        return (self.predict(X) == y).mean()


## 5. Poison crafting + test-time pocket attack (unchanged)

In [5]:
def craft_boundary_pocket_poison(model, X, y, class_id, direction, n_points):
    candidates = np.where(y == class_id)[0]
    proba = model.predict_proba(X[candidates])[:, 1]
    margin = np.abs(proba - 0.5)
    closest = candidates[np.argsort(margin)[:n_points]]
    X_pois = X.copy()
    X_pois[closest] = X[closest] + direction
    return X_pois, closest


def adversarial_attack_pocket(poisoned_model, clean_model, X, y, epsilon_max,
                               step=0.04, max_steps=50, clean_weight=1.0, verbose_every=1000):
    X_adv = X.copy()
    x0 = X.copy()
    success = np.zeros(len(X), dtype=bool)
    active = np.ones(len(X), dtype=bool)

    for s in range(max_steps):
        if not active.any():
            break
        idx = np.where(active)[0]
        Xa = X_adv[idx]
        y_active_np = y[idx]
        y_active = torch.tensor(y_active_np, dtype=torch.long)

        poisoned_model.model.eval()
        xl_p = torch.tensor(Xa, dtype=torch.float32, requires_grad=True)
        proba_p = torch.softmax(poisoned_model.model(xl_p), dim=1)
        proba_p.gather(1, y_active.unsqueeze(1)).squeeze(1).sum().backward()
        grad_p = xl_p.grad.numpy()
        pred_p = proba_p.detach().numpy().argmax(axis=1)

        clean_model.model.eval()
        xl_c = torch.tensor(Xa, dtype=torch.float32, requires_grad=True)
        proba_c = torch.softmax(clean_model.model(xl_c), dim=1)
        proba_c.gather(1, y_active.unsqueeze(1)).squeeze(1).sum().backward()
        grad_c = xl_c.grad.numpy()
        pred_c = proba_c.detach().numpy().argmax(axis=1)

        newly = (pred_p != y_active_np) & (pred_c == y_active_np)
        success[idx[newly]] = True
        active[idx[newly]] = False

        still = ~newly
        idx_s = idx[still]
        if len(idx_s):
            d = -grad_p[still] + clean_weight * grad_c[still]
            nrm = np.linalg.norm(d, axis=1, keepdims=True)
            nrm[nrm < 1e-8] = 1e-8
            d = d / nrm
            X_adv[idx_s] = X_adv[idx_s] + step * d
            delta = X_adv[idx_s] - x0[idx_s]
            dn = np.linalg.norm(delta, axis=1, keepdims=True)
            too_far = dn.ravel() > epsilon_max
            if too_far.any():
                X_adv[idx_s[too_far]] = x0[idx_s[too_far]] + delta[too_far] / dn[too_far] * epsilon_max

        if (s + 1) % verbose_every == 0 or s == max_steps - 1:
            print(f"    step {s+1}/{max_steps}: {success.sum()} succeeded so far, {active.sum()} still active")

    return X_adv, success, np.linalg.norm(X_adv - x0, axis=1)


def honest_adv(X_scaled, X_adv, succ):
    """Perturbed features only where the attack genuinely succeeded; clean everywhere else."""
    out = X_scaled.copy()
    out[succ] = X_adv[succ]
    return out


## 6. Detectors (oracle + meta_al) and unlearning

In [6]:
def build_detector_training_set_mixed(ref_model, X_pois, y, poison_idx_A, poison_idx_B,
                                        n_per_group, uncertain_fraction, rng):
    """`rng` is the ONE shared np.random.Generator for the whole task -- pass the same object in
    everywhere rather than re-deriving a fresh generator from a seed, so randomness continues to
    evolve across every randomized step in the task instead of each step restarting from a fixed
    state."""
    clean_idx = np.setdiff1d(np.arange(len(y)), np.concatenate([poison_idx_A, poison_idx_B]))

    def select_group(pool_idx, n):
        pool_idx = np.asarray(pool_idx)
        n = min(n, len(pool_idx))
        n_uncertain = int(round(uncertain_fraction * n))
        n_random = n - n_uncertain
        proba = ref_model.predict_proba(X_pois[pool_idx])[:, 1]
        margin = np.abs(proba - 0.5)
        order = np.argsort(margin)
        uncertain_pick = pool_idx[order[:n_uncertain]]
        remaining = np.setdiff1d(pool_idx, uncertain_pick)
        random_pick = rng.choice(remaining, size=min(n_random, len(remaining)), replace=False)
        return np.concatenate([uncertain_pick, random_pick])

    clean_benign = select_group(clean_idx[y[clean_idx] == 0], n_per_group)
    clean_malicious = select_group(clean_idx[y[clean_idx] == 1], n_per_group)
    pois_benign = select_group(poison_idx_A, n_per_group)
    pois_malicious = select_group(poison_idx_B, n_per_group)

    idx_all = np.concatenate([clean_benign, clean_malicious, pois_benign, pois_malicious])
    is_poisoned = np.concatenate([
        np.zeros(len(clean_benign)), np.zeros(len(clean_malicious)),
        np.ones(len(pois_benign)), np.ones(len(pois_malicious)),
    ]).astype(int)
    return X_pois[idx_all], is_poisoned, idx_all


def run_oracle_detector(poisoned_model, X_train_poisoned, y_train, idx_pocket_A, idx_pocket_B, rng):
    """`rng` is the task's one shared Generator -- used both for sample-selection (via
    build_detector_training_set_mixed) and to draw the XGBoost model's random_state (XGBoost needs a
    concrete int, so we pull one from the shared stream with rng.integers(...) rather than resetting
    anything to a fixed seed)."""
    X_det, y_det, _ = build_detector_training_set_mixed(
        poisoned_model, X_train_poisoned, y_train, idx_pocket_A, idx_pocket_B,
        n_per_group=N_PER_GROUP, uncertain_fraction=UNCERTAIN_FRACTION, rng=rng,
    )
    detector = XGBClassifier(**xgb_params(int(rng.integers(0, 2**31 - 1)))).fit(X_det, y_det)

    detected = []
    for class_id, true_poison_this_class in [(0, idx_pocket_A), (1, idx_pocket_B)]:
        class_indices = np.where(y_train == class_id)[0]
        preds = detector.predict(X_train_poisoned[class_indices])
        flagged = class_indices[preds == 1]
        tp = len(np.intersect1d(flagged, true_poison_this_class))
        precision = tp / max(len(flagged), 1)
        recall = tp / max(len(true_poison_this_class), 1)
        print(f"    [oracle class={class_id}] flagged {len(flagged)}; precision={precision:.2f} recall={recall:.2f}")
        detected.append(flagged)
    detected_poison_idx = np.concatenate(detected)
    print(f"    oracle total detected: {len(detected_poison_idx)}")
    return detected_poison_idx


def get_latent_features(model, X, batch_size=512):
    model.eval()
    captured = {}
    def _hook(module, inp, out):
        captured["z"] = out.detach()
    handle = model.bns[-1].register_forward_hook(_hook)
    latents = []
    with torch.no_grad():
        Xt = torch.as_tensor(np.asarray(X), dtype=torch.float32)
        for i in range(0, len(Xt), batch_size):
            _ = model(Xt[i:i + batch_size])
            latents.append(torch.relu(captured["z"]).numpy())
    handle.remove()
    return np.concatenate(latents, axis=0)


def replay_knn_distance(X, y, replay_X_np, replay_y_np, k=5):
    dist = np.full(len(X), np.nan)
    for c in np.unique(y):
        mask = y == c
        replay_mask = replay_y_np == c
        if replay_mask.sum() == 0:
            continue
        kk = min(k, replay_mask.sum())
        nn = NearestNeighbors(n_neighbors=kk).fit(replay_X_np[replay_mask])
        d, _ = nn.kneighbors(X[mask])
        dist[mask] = d.mean(axis=1)
    return dist


def fit_density_ratio_model(X_current, replay_X_np):
    Xd = np.vstack([replay_X_np, X_current])
    yd = np.concatenate([np.zeros(len(replay_X_np)), np.ones(len(X_current))])
    return LogisticRegression(max_iter=1000, class_weight="balanced", random_state=0).fit(Xd, yd)


def density_ratio_score(clf, X):
    p = np.clip(clf.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)
    return p / (1 - p)


def extract_meta_features(adaptable_model, X, y, replay_X_np, replay_y_np, k=5):
    net = adaptable_model.model
    net.eval()
    Xt = torch.as_tensor(np.asarray(X), dtype=torch.float32)
    yt = torch.as_tensor(np.asarray(y), dtype=torch.long)
    latent = get_latent_features(net, X)

    iso_raw = IsolationForest(n_estimators=200, contamination="auto", random_state=0).fit(X)
    iso_latent = IsolationForest(n_estimators=200, contamination="auto", random_state=0).fit(latent)
    density_clf = fit_density_ratio_model(X, replay_X_np)

    iso_raw_score = -iso_raw.score_samples(X)
    iso_latent_score = -iso_latent.score_samples(latent)
    density_ratio = density_ratio_score(density_clf, X)

    with torch.no_grad():
        logits = net(Xt)
        proba = torch.softmax(logits, dim=1)
        per_sample_loss = nn.functional.cross_entropy(logits, yt, reduction="none").numpy()
        p = proba.numpy()
        eps_ = 1e-12
        entropy = -(p * np.log(p + eps_)).sum(axis=1)
        sorted_p = np.sort(p, axis=1)
        top2_margin = sorted_p[:, -1] - sorted_p[:, -2]

    dist_to_replay = replay_knn_distance(X, y, replay_X_np, replay_y_np, k=k)

    feats = np.column_stack([iso_raw_score, iso_latent_score, per_sample_loss, entropy, top2_margin,
                              density_ratio, dist_to_replay])
    return feats


def run_meta_al_detector(poisoned_model, X_train_poisoned, y_train, poison_idx, replay_X, replay_y, rng):
    """Active-learning episodic sampling: same ~250-label budget as the original random-sampling
    meta detector, but each inner step ALWAYS queries SAMPLES_PER_INNER_STEP // 2 points from the
    poison pool and the rest from the clean pool -- never letting the split drift -- picking within
    each pool the CURRENT detector's least-confident untouched points (once a detector is trainable).
    No test-time access, no borrowed oracle points. `rng` is the task's one shared Generator, same as
    run_oracle_detector."""
    replay_y_np = np.asarray(replay_y)
    feats_meta = extract_meta_features(poisoned_model, X_train_poisoned, y_train, replay_X, replay_y_np)
    is_poisoned_true = np.zeros(len(y_train), dtype=int)
    is_poisoned_true[poison_idx] = 1
    feats_meta_std = StandardScaler().fit_transform(feats_meta)

    poison_pool = poison_idx
    clean_pool = np.setdiff1d(np.arange(len(y_train)), poison_idx)

    n_p = SAMPLES_PER_INNER_STEP // 2
    n_c = SAMPLES_PER_INNER_STEP - n_p

    touched_al, labels_al = set(), {}

    seed_p = rng.choice(poison_pool, n_p, replace=False)
    seed_c = rng.choice(clean_pool, n_c, replace=False)
    for i in np.concatenate([seed_p, seed_c]):
        touched_al.add(int(i)); labels_al[int(i)] = int(is_poisoned_true[i])

    for ep in range(N_OUTER_EPISODES):
        for step in range(N_INNER_STEPS):
            idx_touched = np.array(sorted(touched_al))
            y_touched = np.array([labels_al[i] for i in idx_touched])

            untouched_poison = np.setdiff1d(poison_pool, idx_touched)
            untouched_clean = np.setdiff1d(clean_pool, idx_touched)

            if len(np.unique(y_touched)) < 2:
                # no AL detector trainable yet -- random draw within each pool, still split n_p/n_c
                idx_p = rng.choice(untouched_poison, min(n_p, len(untouched_poison)), replace=False)
                idx_c = rng.choice(untouched_clean, min(n_c, len(untouched_clean)), replace=False)
            else:
                al_detector = XGBClassifier(**xgb_params(int(rng.integers(0, 2**31 - 1)))).fit(
                    feats_meta_std[idx_touched], y_touched)
                # rank uncertainty SEPARATELY within each pool so the split stays fixed regardless of
                # which class the detector currently finds more ambiguous overall
                margin_p = np.abs(al_detector.predict_proba(feats_meta_std[untouched_poison])[:, 1] - 0.5)
                margin_c = np.abs(al_detector.predict_proba(feats_meta_std[untouched_clean])[:, 1] - 0.5)
                idx_p = untouched_poison[np.argsort(margin_p)[:min(n_p, len(untouched_poison))]]
                idx_c = untouched_clean[np.argsort(margin_c)[:min(n_c, len(untouched_clean))]]

            new_idx = np.concatenate([idx_p, idx_c])
            for i in new_idx:
                touched_al.add(int(i)); labels_al[int(i)] = int(is_poisoned_true[i])

    idx_touched_al = np.array(sorted(touched_al))
    y_touched_al = np.array([labels_al[i] for i in idx_touched_al])
    print(f"    active-learning budget used: {len(idx_touched_al)} labels "
          f"(poisoned: {y_touched_al.sum()}, clean: {(y_touched_al==0).sum()})")

    xgb_meta_al = XGBClassifier(**xgb_params(int(rng.integers(0, 2**31 - 1)))).fit(
        feats_meta_std[idx_touched_al], y_touched_al)

    query_mask_al = np.ones(len(is_poisoned_true), dtype=bool)
    query_mask_al[idx_touched_al] = False
    Xq_al, yq_al = feats_meta_std[query_mask_al], is_poisoned_true[query_mask_al]
    preds_al = xgb_meta_al.predict(Xq_al)
    scores_al = xgb_meta_al.predict_proba(Xq_al)[:, 1]
    prec_al, rec_al, f1_al, _ = precision_recall_fscore_support(yq_al, preds_al, average="binary", zero_division=0)
    auc_al = roc_auc_score(yq_al, scores_al) if len(np.unique(yq_al)) > 1 else float("nan")
    print(f"    [meta_al] held-out ({query_mask_al.sum()} pts): precision={prec_al:.3f} recall={rec_al:.3f} "
          f"f1={f1_al:.3f} auc={auc_al:.3f}")

    meta_al_detected = np.where(query_mask_al)[0][preds_al == 1]
    return meta_al_detected

def unlearn_dropped_rows(start_model, detected_idx, X_train_poisoned, y_train, replay_X, replay_y,
                          epochs=20):
    """Continues from `start_model` (the SAME lineage's own state from the previous task) -- trains
    on this task's own data with the flagged points dropped. No manual_seed call here -- torch's global
    RNG is seeded ONCE per task (in run_task_pipeline) and left to evolve naturally through every step
    that follows, rather than being reset before each one."""
    mask_keep = np.ones(len(X_train_poisoned), dtype=bool)
    mask_keep[detected_idx] = False
    X_clean_only, y_clean_only = X_train_poisoned[mask_keep], y_train[mask_keep]
    model = AdaptableClassifier(copy.deepcopy(start_model.model))
    model.adapt(X_clean_only, y_clean_only, replay_X=replay_X, replay_y=replay_y, epochs=epochs)
    return model


def unlearn_amnesiac(start_model, detected_idx, X_train_poisoned, y_train, replay_X, replay_y,
                      epochs=20):
    mask_keep = np.ones(len(X_train_poisoned), dtype=bool)
    mask_keep[detected_idx] = False
    X_clean_only, y_clean_only = X_train_poisoned[mask_keep], y_train[mask_keep]
    X_flagged = X_train_poisoned[detected_idx]
    y_flagged = y_train[detected_idx]
    X_flagged_dup = np.vstack([X_flagged, X_flagged])
    y_flagged_dup = np.concatenate([np.zeros(len(X_flagged)), np.ones(len(X_flagged))])
    model = AdaptableClassifier(copy.deepcopy(start_model.model))
    for _ in range(epochs):
        model.adapt(X_flagged_dup, y_flagged_dup, epochs=1)
        model.adapt(X_clean_only, y_clean_only, replay_X=replay_X, replay_y=replay_y, epochs=1)
    return model


def unlearn_opposite_class(start_model, detected_idx, X_train_poisoned, y_train, replay_X, replay_y,
                            epochs=20):
    y_relabel = y_train.copy()
    y_relabel[detected_idx] = 1 - y_train[detected_idx]
    model = AdaptableClassifier(copy.deepcopy(start_model.model))
    model.adapt(X_train_poisoned, y_relabel, replay_X=replay_X, replay_y=replay_y, epochs=epochs)
    return model


def test_time_adapt(start_model, flagged_idx, X_probe, y_probe, X_train_poisoned, y_train,
                     replay_X, replay_y, epochs=3, lr=1e-4):
    """Fixed version: retrains on the FULL kept-training set (not just probe+replay), so the
    model keeps its anchor to what it already learned instead of catastrophically collapsing
    clean accuracy."""
    mask_keep = np.ones(len(X_train_poisoned), dtype=bool)
    mask_keep[flagged_idx] = False
    tta_model = AdaptableClassifier(copy.deepcopy(start_model.model), lr=lr)
    X_combined = np.vstack([X_train_poisoned[mask_keep], X_probe])
    y_combined = np.concatenate([y_train[mask_keep], y_probe])
    tta_model.adapt(X_combined, y_combined, replay_X=replay_X, replay_y=replay_y, epochs=epochs)
    return tta_model


## 7. Per-task driver functions -- CHAINED continual-learning design, Task 0 trains from scratch

**Task 0** (`run_task0_pipeline`) trains a fresh classifier on task 0's own data only -- no replay
buffer, no poisoning, matching how the real pipeline never poisoned task 0. Its single trained model
becomes the seed for ALL 8 lineages entering Task 1 (`clean_model`, `poisoned_model`, and the 6 fixed
lineages) -- there's no distinction between them yet since nothing has been poisoned.

**Tasks 1-9** (`run_task_pipeline`) each continue every lineage from **that same lineage's own**
previous-task state:

- **`clean_model`** continues from the previous task's `clean_model`, adapted on this task's own clean
  training data.
- **`poisoned_model`** (`"poisoned (baseline / MADAR)"`) continues from the previous task's
  `poisoned_model`, adapted on this task's freshly-poisoned training data -- poison compounds across
  tasks for this undefended lineage.
- **The 6 fixed lineages** each continue from their own previous task's final model -- a fix applied at
  Task 1 has to survive Task 2's adaptation, Task 2's new poison, and Task 2's fresh detection+unlearning
  pass, and so on through Task 9.
- The **replay buffer** is read from the real checkpoint chain each task (`ckpts[task_id - 1]`, i.e. the
  end-of-previous-task buffer state) -- not self-tracked, since reservoir sampling isn't reconstructed
  here.
- Detection (oracle + meta_al) and the calibration/TTA probe are computed **fresh each task**, against
  that task's own newly-injected poison.

`state` is a plain dict carried across task cells within one architecture's run: it caches loaded
checkpoints and holds the previous task's live model objects so the next task's call can continue from
them. A fresh `state` is created for each of the 4 architecture runs.


In [7]:
def save_task_models(task_id, models_dict, seeds_dict, extra_meta=None):
    task_dir = MODEL_SAVE_DIR / f"task{task_id}"
    task_dir.mkdir(parents=True, exist_ok=True)
    for name, model in models_dict.items():
        fname = (name.replace(" ", "_").replace("(", "").replace(")", "")
                     .replace("/", "-").replace("+", "plus"))
        torch.save(model.model.state_dict(), task_dir / f"{fname}.pt")
    meta = {"seeds": seeds_dict}
    if extra_meta:
        meta.update(extra_meta)
    with open(task_dir / "run_meta.json", "w") as f:
        json.dump(meta, f, indent=2, default=str)
    print(f"  saved {len(models_dict)} models + run_meta.json to {task_dir}")


In [8]:
def run_task0_pipeline(state, hidden_sizes, epochs=TASK0_EPOCHS):
    """Task 0: trained FROM SCRATCH on task 0's own data -- no replay buffer (no prior task to replay
    from) and no poisoning (the real pipeline never poisoned task 0 either -- task0.pt's replay buffer
    has 0% *_perturbed entries). hidden_sizes fixes the architecture for this whole run: every later
    task just deep-copies this model's structure forward, so it's never passed again after this."""
    print(f"\n{'='*100}\nTASK 0  (hidden_sizes={hidden_sizes})\n{'='*100}")
    state.setdefault("ckpts_all", {})
    state.setdefault("results", {})
    state["hidden_sizes"] = hidden_sizes

    task_seed = SEED
    torch.manual_seed(task_seed)
    rng = np.random.default_rng(task_seed)

    ckpts = get_ckpts_through(0, state["ckpts_all"])
    task_train_gids, task_test_gids_final, INDEX_OFFSET = reconstruct_task_gids(0, ckpts)

    X_train_raw, y_train, gid_train = rows_for(task_train_gids[0])
    print(f"Task 0: X_train={X_train_raw.shape}")

    ckpt0 = ckpts[0]
    scaler = ckpt0["scaler"]
    X_train_scaled = scaler.transform(X_train_raw.astype("float32"))

    X_test_raw, y_test, _ = rows_for(task_test_gids_final[0])
    X_test_scaled = scaler.transform(X_test_raw.astype("float32"))
    test_data = {0: (X_test_scaled, y_test)}

    feature_dim = ckpt0["feature_dim"]
    net = ClassifierNN(feature_dim, num_classes=2, hidden_sizes=hidden_sizes)
    model0 = AdaptableClassifier(copy.deepcopy(net))
    model0.adapt(X_train_scaled, y_train, epochs=epochs)   # no replay_X/replay_y passed at all
    t0_acc = model0.score(*test_data[0])
    print(f"[task0-trained, {len(hidden_sizes)}-hidden-layer] task 0 test acc: {t0_acc:.3f}")

    y_pred0 = model0.predict(X_test_scaled)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred0, average="binary", zero_division=0)

    final_df = pd.DataFrame([{
        "model": "task0-trained (clean, no poison)",
        "t0_clean": t0_acc, "t0_adv": float("nan"), "t0_combined": t0_acc,
        "t0_precision": prec, "t0_recall": rec, "t0_f1": f1,
        "mean_combined": t0_acc,
    }])

    # this single model seeds ALL 8 lineages entering Task 1 -- task 0 has no poison yet, so there's
    # no distinction between clean/poisoned/fixed lineages until Task 1's own poisoning happens
    models_to_save = {"clean_model": model0, "poisoned_model": model0,
                       **{k: model0 for k in ["dropped_oracle", "amnesiac_oracle", "opposite_oracle",
                                               "dropped_al_tta", "amnesiac_al_tta", "opposite_al_tta"]}}
    save_task_models(0, {"clean_model": model0}, {"task_seed": task_seed})

    result = dict(final_df=final_df, test_data=test_data, models=models_to_save)
    state["results"][0] = result
    return result


In [9]:
def run_task_pipeline(task_id, state):
    """state: dict persisted across task cells.
        ckpts_all         -- {task_id: raw checkpoint dict}, cached across calls
        results[t]        -- everything returned from task t, including the live model objects that
                              the NEXT task's cell continues from (results[0] comes from
                              run_task0_pipeline, everything else from this function)
    """
    print(f"\n{'='*100}\nTASK {task_id}\n{'='*100}")
    state.setdefault("ckpts_all", {})
    state.setdefault("results", {})

    base_task_id = task_id - 1
    task_seed = SEED
    torch.manual_seed(task_seed)
    rng = np.random.default_rng(task_seed)

    ckpts = get_ckpts_through(task_id, state["ckpts_all"])
    task_train_gids, task_test_gids_final, INDEX_OFFSET = reconstruct_task_gids(task_id, ckpts)

    X_train_raw, y_train, gid_train = rows_for(task_train_gids[task_id])
    print(f"Task {task_id}: X_train={X_train_raw.shape}")

    base_ckpt = ckpts[base_task_id]
    scaler = base_ckpt["scaler"]
    X_train_scaled = scaler.transform(X_train_raw.astype("float32"))

    replay_X_raw, replay_y, replay_cat, replay_gid = reconstruct_replay_buffer(
        base_ckpt["baseline_replay_buffer"], offset=INDEX_OFFSET)
    replay_X = scaler.transform(replay_X_raw)
    print(f"replay buffer (end of task {base_task_id}, from checkpoint_task{base_task_id}.pt): {replay_X.shape}")

    test_data = {}
    for t in range(task_id + 1):
        X_t_raw, y_t, _ = rows_for(task_test_gids_final[t])
        test_data[t] = (scaler.transform(X_t_raw.astype("float32")), y_t)
        print(f"  test set task {t}: {test_data[t][0].shape}")

    # every task from 1 onward (including task 1) chains from the PREVIOUS task's own live models --
    # task 1 reads state["results"][0], which run_task0_pipeline populated
    prev_models = state["results"][task_id - 1]["models"]
    prev_clean = prev_models["clean_model"]
    prev_poisoned = prev_models["poisoned_model"]
    prev_fixed = {k: prev_models[k] for k in
                  ["dropped_oracle", "amnesiac_oracle", "opposite_oracle",
                   "dropped_al_tta", "amnesiac_al_tta", "opposite_al_tta"]}

    clean_model = AdaptableClassifier(copy.deepcopy(prev_clean.model))
    clean_model.adapt(X_train_scaled, y_train, replay_X=replay_X, replay_y=replay_y,
                       epochs=CLEAN_ADAPT_EPOCHS)
    print(f"[clean-adapted] task {task_id} test acc: {clean_model.score(*test_data[task_id]):.3f}")

    pocket_toward_1 = X_train_scaled[y_train == 1].mean(axis=0)
    pocket_toward_0 = X_train_scaled[y_train == 0].mean(axis=0)
    separating_direction = pocket_toward_1 - pocket_toward_0

    n_class0 = (y_train == 0).sum()
    n_class1 = (y_train == 1).sum()
    n_poison_0 = int(POISON_FRACTION * n_class0)
    n_poison_1 = int(POISON_FRACTION * n_class1)

    X_p1, idx_pocket_A = craft_boundary_pocket_poison(
        clean_model, X_train_scaled, y_train, class_id=0, direction=separating_direction, n_points=n_poison_0)
    X_train_poisoned, idx_pocket_B = craft_boundary_pocket_poison(
        clean_model, X_p1, y_train, class_id=1, direction=-separating_direction, n_points=n_poison_1)
    poison_idx = np.concatenate([idx_pocket_A, idx_pocket_B])
    print(f"poisoned {len(poison_idx)}/{len(X_train_scaled)} = {len(poison_idx)/len(X_train_scaled):.1%} of task {task_id}'s train set")

    poisoned_model = AdaptableClassifier(copy.deepcopy(prev_poisoned.model))
    poisoned_model.adapt(X_train_poisoned, y_train, replay_X=replay_X, replay_y=replay_y, epochs=ADAPT_EPOCHS)
    print(f"[poisoned-adapted] clean task {task_id} test acc: {poisoned_model.score(*test_data[task_id]):.3f}")

    typical_gap = np.linalg.norm(test_data[task_id][0][test_data[task_id][1] == 0] - pocket_toward_1, axis=1).mean()
    EPS_TEST = typical_gap * 0.15

    adv_honest, succ_by_task = {}, {}
    for t in range(task_id + 1):
        Xt, yt = test_data[t]
        print(f"  attacking task {t}'s test set ({len(yt)} pts)...")
        X_adv_t, succ_t, _ = adversarial_attack_pocket(poisoned_model, clean_model, Xt, yt, epsilon_max=EPS_TEST)
        adv_honest[t] = honest_adv(Xt, X_adv_t, succ_t)
        succ_by_task[t] = succ_t
        print(f"    task {t} genuine pockets: {succ_t.sum()}/{len(yt)} ({succ_t.mean():.1%})")

    print("running oracle detector...")
    detected_poison_idx = run_oracle_detector(poisoned_model, X_train_poisoned, y_train,
                                               idx_pocket_A, idx_pocket_B, rng=rng)
    print("running meta_al detector...")
    meta_al_detected = run_meta_al_detector(poisoned_model, X_train_poisoned, y_train, poison_idx,
                                             replay_X, replay_y, rng=rng)

    dropped_oracle = unlearn_dropped_rows(prev_fixed["dropped_oracle"], detected_poison_idx,
                                           X_train_poisoned, y_train, replay_X, replay_y)
    amnesiac_oracle = unlearn_amnesiac(prev_fixed["amnesiac_oracle"], detected_poison_idx,
                                        X_train_poisoned, y_train, replay_X, replay_y)
    opposite_oracle = unlearn_opposite_class(prev_fixed["opposite_oracle"], detected_poison_idx,
                                              X_train_poisoned, y_train, replay_X, replay_y)
    print(f"  [oracle] dropped/amnesiac/opposite each continued from their own prior state, "
          f"fixed on {len(detected_poison_idx)} newly-flagged points")

    dropped_al = unlearn_dropped_rows(prev_fixed["dropped_al_tta"], meta_al_detected,
                                       X_train_poisoned, y_train, replay_X, replay_y)
    amnesiac_al = unlearn_amnesiac(prev_fixed["amnesiac_al_tta"], meta_al_detected,
                                    X_train_poisoned, y_train, replay_X, replay_y)
    opposite_al = unlearn_opposite_class(prev_fixed["opposite_al_tta"], meta_al_detected,
                                          X_train_poisoned, y_train, replay_X, replay_y)
    print(f"  [meta_al] dropped/amnesiac/opposite each continued from their own prior state, "
          f"fixed on {len(meta_al_detected)} newly-flagged points")

    calib_X_list, calib_y_list, eval_pockets = [], [], {}
    for t in range(task_id + 1):
        Xt, yt = test_data[t]
        succ_t = succ_by_task[t]
        pocket_indices = np.where(succ_t)[0]
        n_calib = int(CALIB_FRAC * len(pocket_indices))
        calib_idx_t = rng.choice(pocket_indices, n_calib, replace=False) if n_calib > 0 else np.array([], dtype=int)
        eval_idx_t = np.setdiff1d(pocket_indices, calib_idx_t)
        calib_X_list.append(adv_honest[t][calib_idx_t])
        calib_y_list.append(yt[calib_idx_t])
        eval_pockets[t] = (adv_honest[t][eval_idx_t], yt[eval_idx_t])
        print(f"  task {t}: calib pockets={len(calib_idx_t)}  held-out eval pockets={len(eval_idx_t)}")

    X_calib_combined = np.vstack(calib_X_list)
    y_calib_combined = np.concatenate(calib_y_list)
    print(f"combined calibration probe across tasks 0..{task_id}: {len(y_calib_combined)} points")

    dropped_al_tta = test_time_adapt(dropped_al, meta_al_detected, X_calib_combined, y_calib_combined,
                                      X_train_poisoned, y_train, replay_X, replay_y)
    amnesiac_al_tta = test_time_adapt(amnesiac_al, meta_al_detected, X_calib_combined, y_calib_combined,
                                       X_train_poisoned, y_train, replay_X, replay_y)
    opposite_al_tta = test_time_adapt(opposite_al, meta_al_detected, X_calib_combined, y_calib_combined,
                                       X_train_poisoned, y_train, replay_X, replay_y)

    final_models = [
        ("poisoned (baseline / MADAR)", poisoned_model),
        ("dropped-rows (oracle)", dropped_oracle),
        ("amnesiac (oracle)", amnesiac_oracle),
        ("opposite-class (oracle)", opposite_oracle),
        ("dropped-rows (meta_al+TTA)", dropped_al_tta),
        ("amnesiac (meta_al+TTA)", amnesiac_al_tta),
        ("opposite-class (meta_al+TTA)", opposite_al_tta),
    ]

    header = f"{'model':<32}"
    for t in range(task_id + 1):
        header += f" {'T'+str(t)+' clean':>9} {'T'+str(t)+' adv':>8} {'T'+str(t)+' comb':>9} {'T'+str(t)+' f1':>7}"
    header += f" {'MEAN COMB':>10}"
    print("\n" + header)
    print("=" * len(header))

    final_rows = []
    for name, model in final_models:
        row = {"model": name}
        combined_accs = []
        line = f"{name:<32}"
        for t in range(task_id + 1):
            Xt, yt = test_data[t]
            Xp, yp = eval_pockets[t]
            clean_acc = model.score(Xt, yt)
            adv_acc = model.score(Xp, yp) if len(yp) else float("nan")
            if len(yp):
                X_eval, y_eval = np.vstack([Xt, Xp]), np.concatenate([yt, yp])
            else:
                X_eval, y_eval = Xt, yt
            comb_acc = model.score(X_eval, y_eval)
            y_pred_eval = model.predict(X_eval)
            prec, rec, f1, _ = precision_recall_fscore_support(y_eval, y_pred_eval, average="binary", zero_division=0)
            combined_accs.append(comb_acc)
            row[f"t{t}_clean"] = clean_acc
            row[f"t{t}_adv"] = adv_acc
            row[f"t{t}_combined"] = comb_acc
            row[f"t{t}_precision"] = prec
            row[f"t{t}_recall"] = rec
            row[f"t{t}_f1"] = f1
            line += f" {clean_acc:>9.3f} {adv_acc:>8.3f} {comb_acc:>9.3f} {f1:>7.3f}"
        mean_combined = float(np.mean(combined_accs))
        row["mean_combined"] = mean_combined
        line += f" {mean_combined:>10.3f}"
        print(line)
        final_rows.append(row)

    final_df = pd.DataFrame(final_rows)

    models_to_save = {
        "clean_model": clean_model,
        "poisoned_model": poisoned_model,
        "dropped_oracle": dropped_oracle, "amnesiac_oracle": amnesiac_oracle, "opposite_oracle": opposite_oracle,
        "dropped_al_tta": dropped_al_tta, "amnesiac_al_tta": amnesiac_al_tta, "opposite_al_tta": opposite_al_tta,
    }
    seeds_used = {"task_seed": task_seed}
    save_task_models(task_id, models_to_save, seeds_used,
                      extra_meta={"n_oracle_detected": int(len(detected_poison_idx)),
                                  "n_meta_al_detected": int(len(meta_al_detected)),
                                  "n_calib_combined": int(len(y_calib_combined))})

    result = dict(final_df=final_df, test_data=test_data, adv_honest=adv_honest, succ_by_task=succ_by_task,
                  eval_pockets=eval_pockets, X_calib_combined=X_calib_combined, y_calib_combined=y_calib_combined,
                  detected_poison_idx=detected_poison_idx, meta_al_detected=meta_al_detected,
                  models=models_to_save)
    state["results"][task_id] = result
    return result


## 8. Run Task 0-9 for all 4 architectures

Each architecture (1, 2, 3, 4 hidden layers) runs Task 0-9 in its **own cell** below, so you can run,
inspect, or re-run any single architecture independently without repeating the others. Each cell calls
`run_one_architecture(depth)` (defined once, just below) with its own fresh `state` and its own
`MODEL_SAVE_DIR` -- its `final_df`s (including precision/recall/f1) get stacked into one table, written
to `RESULTS_DIR`, and stored in `all_architecture_results[depth]`. The last cell combines all 4 into one
CSV -- run it only after whichever of the 4 architecture cells above it you want included have finished.

If you want to sanity-check first, just run the "depth = 1" cell alone before running the rest.


In [10]:
all_architecture_results = {}


def run_one_architecture(depth):
    """Runs Task 0-9 for HIDDEN_SIZES_BY_DEPTH[depth] with a fresh state, saves models under their own
    MODEL_SAVE_DIR, writes this architecture's p/r/f1 table to RESULTS_DIR, and returns that table."""
    hidden_sizes = HIDDEN_SIZES_BY_DEPTH[depth]
    print(f"\n{'#'*100}\n RUNNING {depth}-HIDDEN-LAYER ARCHITECTURE {hidden_sizes}\n{'#'*100}")
    global MODEL_SAVE_DIR
    MODEL_SAVE_DIR = MODEL_SAVE_DIR_BASE / f"seed{SEED}_depth{depth}"
    state = {}
    run_task0_pipeline(state, hidden_sizes)
    for t in range(1, 10):
        run_task_pipeline(t, state)

    # ---- stack every task's final_df (with p/r/f1) into one table for this architecture ----
    rows = []
    for t in range(0, 10):
        df = state["results"][t]["final_df"]
        for _, row in df.iterrows():
            row_dict = row.to_dict()
            row_dict["task"] = t
            row_dict["depth"] = depth
            rows.append(row_dict)
    depth_df = pd.DataFrame(rows)
    cols = ["depth", "task", "model"] + [c for c in depth_df.columns if c not in ("depth", "task", "model")]
    depth_df = depth_df[cols]

    out_path = RESULTS_DIR / f"prf_report_seed{SEED}_depth{depth}.csv"
    depth_df.to_csv(out_path, index=False)
    print(f"saved {len(depth_df)} rows to {out_path}")
    return depth_df


### depth = 1  (1 hidden layer)

In [ ]:
all_architecture_results[1] = run_one_architecture(1)
all_architecture_results[1]


### depth = 2  (2 hidden layers)

In [ ]:
all_architecture_results[2] = run_one_architecture(2)
all_architecture_results[2]


### depth = 3  (3 hidden layers)

In [24]:
all_architecture_results[3] = run_one_architecture(3)
all_architecture_results[3]



####################################################################################################
 RUNNING 3-HIDDEN-LAYER ARCHITECTURE (512, 256, 128)
####################################################################################################

TASK 0  (hidden_sizes=(512, 256, 128))
  INDEX_OFFSET = 0  (out-of-bounds counts: {0: np.int64(0), 1: np.int64(0)})
  task 0: train=56162 test=14038
Task 0: X_train=(56162, 1488)


/home/aritran/.local/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[task0-trained, 3-hidden-layer] task 0 test acc: 0.965
  saved 1 models + run_meta.json to saved_models_datasensitivity/seed100_depth3/task0

TASK 1
  INDEX_OFFSET = 0  (out-of-bounds counts: {0: np.int64(0), 1: np.int64(0)})
  task 0: train=56162 test=14038
  task 1: train=17185 test=4296
Task 1: X_train=(17185, 1488)
replay buffer (end of task 0, from checkpoint_task0.pt): (4000, 1488)
  test set task 0: (14038, 1488)
  test set task 1: (4296, 1488)


/home/aritran/.local/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[clean-adapted] task 1 test acc: 0.971
poisoned 5155/17185 = 30.0% of task 1's train set
[poisoned-adapted] clean task 1 test acc: 0.872
  attacking task 0's test set (14038 pts)...
    step 50/50: 11779 succeeded so far, 2259 still active
    task 0 genuine pockets: 11779/14038 (83.9%)
  attacking task 1's test set (4296 pts)...
    step 50/50: 3482 succeeded so far, 814 still active
    task 1 genuine pockets: 3482/4296 (81.1%)
running oracle detector...
    [oracle class=0] flagged 0; precision=0.00 recall=0.00
    [oracle class=1] flagged 0; precision=0.00 recall=0.00
    oracle total detected: 0
running meta_al detector...
    active-learning budget used: 100 labels (poisoned: 50, clean: 50)
    [meta_al] held-out (17085 pts): precision=0.974 recall=0.886 f1=0.928 auc=0.962
  [oracle] dropped/amnesiac/opposite each continued from their own prior state, fixed on 0 newly-flagged points
  [meta_al] dropped/amnesiac/opposite each continued from their own prior state, fixed on 4648 new

/home/aritran/.local/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


... (12 outputs left out)

  test set task 3: (3968, 1488)
  test set task 4: (3804, 1488)
  test set task 5: (3641, 1488)
  test set task 6: (3477, 1488)
  test set task 7: (3313, 1488)
  test set task 8: (3149, 1488)
[clean-adapted] task 8 test acc: 0.930
poisoned 3789/12632 = 30.0% of task 8's train set
[poisoned-adapted] clean task 8 test acc: 0.886
  attacking task 0's test set (14038 pts)...
    step 50/50: 11421 succeeded so far, 2617 still active
    task 0 genuine pockets: 11421/14038 (81.4%)
  attacking task 1's test set (4296 pts)...
    step 50/50: 3524 succeeded so far, 772 still active
    task 1 genuine pockets: 3524/4296 (82.0%)
  attacking task 2's test set (4132 pts)...
    step 50/50: 3376 succeeded so far, 756 still active
    task 2 genuine pockets: 3376/4132 (81.7%)
  attacking task 3's test set (3968 pts)...
    step 50/50: 3225 succeeded so far, 743 still active
    task 3 genuine pockets: 3225/3968 (81.3%)
  attacking task 4's test set (3804 pts)...
    step 50/50: 3131 succeeded so far,

/home/aritran/.local/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  test set task 3: (3968, 1488)
  test set task 4: (3804, 1488)
  test set task 5: (3641, 1488)
  test set task 6: (3477, 1488)
  test set task 7: (3313, 1488)
  test set task 8: (3149, 1488)
  test set task 9: (2986, 1488)
[clean-adapted] task 9 test acc: 0.938
poisoned 3580/11934 = 30.0% of task 9's train set
[poisoned-adapted] clean task 9 test acc: 0.860
  attacking task 0's test set (14038 pts)...
    step 50/50: 11894 succeeded so far, 2144 still active
    task 0 genuine pockets: 11894/14038 (84.7%)
  attacking task 1's test set (4296 pts)...
    step 50/50: 3671 succeeded so far, 625 still active
    task 1 genuine pockets: 3671/4296 (85.5%)
  attacking task 2's test set (4132 pts)...
    step 50/50: 3540 succeeded so far, 592 still active
    task 2 genuine pockets: 3540/4132 (85.7%)
  attacking task 3's test set (3968 pts)...
    step 50/50: 3379 succeeded so far, 589 still active
    task 3 genuine pockets: 3379/3968 (85.2%)
  attacking task 4's test set (3804 pts)...
    st

,depth,task,model,t0_clean,t0_adv,t0_combined,t0_precision,t0_recall,t0_f1,mean_combined,...,t8_combined,t8_precision,t8_recall,t8_f1,t9_clean,t9_adv,t9_combined,t9_precision,t9_recall,t9_f1
0,3,0,"task0-trained (clean, no poison)",0.964952,NaN,0.964952,0.923424,1.000000,0.960188,0.964952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,1,poisoned (baseline / MADAR),0.769697,0.000000,0.421395,0.347393,0.295881,0.319574,0.453061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,dropped-rows (oracle),0.852472,0.618288,0.746500,0.682135,0.838896,0.752438,0.781352,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1,amnesiac (oracle),0.813720,0.631906,0.731446,0.760247,0.606454,0.674698,0.760494,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,1,opposite-class (oracle),0.862730,0.534603,0.714247,0.643023,0.849172,0.731857,0.732544,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,3,9,amnesiac (oracle),0.812580,0.713042,0.767298,0.723469,0.783223,0.752161,0.809051,...,0.856776,0.764531,0.982382,0.859873,0.868386,0.817649,0.844961,0.762477,0.973139,0.855024
60,3,9,opposite-class (oracle),0.773258,0.597644,0.693368,0.625236,0.798467,0.701312,0.742211,...,0.783964,0.678383,0.983148,0.802815,0.832887,0.731745,0.786191,0.691996,0.981965,0.811865
61,3,9,dropped-rows (meta_al+TTA),0.865080,0.851741,0.859012,0.784553,0.947464,0.858347,0.833552,...,0.849923,0.769326,0.949062,0.849794,0.844943,0.822725,0.834685,0.769035,0.926324,0.840383
62,3,9,amnesiac (meta_al+TTA),0.856888,0.824684,0.842238,0.763879,0.940918,0.843206,0.823996,...,0.841186,0.744200,0.982765,0.847004,0.853315,0.824287,0.839913,0.751611,0.984651,0.852492


### depth = 4  (4 hidden layers -- matches the real checkpoints)

In [ ]:
all_architecture_results[4] = run_one_architecture(4)
all_architecture_results[4]


## 9. Combined table across whichever architectures you ran above


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

combined_all_depths = pd.concat(all_architecture_results.values(), ignore_index=True)
combined_out_path = RESULTS_DIR / f"prf_report_seed{SEED}_all_depths.csv"
combined_all_depths.to_csv(combined_out_path, index=False)
print(f"saved combined {len(combined_all_depths)} rows to {combined_out_path}")
combined_all_depths
